<a href="https://colab.research.google.com/github/Dshah1003/CS4375-MLProject/blob/main/CS_4375_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Dependencies
!pip install datasets transformers torch pandas scikit-learn streamlit -q

In [ ]:
# Load Data
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("domenicrosati/TruthfulQA", split="train")
df = pd.DataFrame(dataset)

df = df[["Question", "Best Answer", "Correct Answers", "Incorrect Answers", "Category"]]
df.rename(columns={
    "Question": "question",
    "Best Answer": "best_answer",
    "Correct Answers": "correct_answers",
    "Incorrect Answers": "incorrect_answers",
    "Category": "category"
}, inplace=True)

print(f"Loaded {len(df)} questions across {df['category'].nunique()} categories.")
df.head()

In [ ]:
# Load Flan-T5 Model
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

model_name = "google/flan-t5-base"
print(f"Loading model: {model_name}...")

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"✅ Model loaded on {device}.")

In [ ]:
# Generated Answer from Flan-T5
def generate_answer(question):
    prompt = f"Answer this question truthfully and to the best of your ability: {question}"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Create Copy DataFrame
sample_df = df.copy()

answers = []
for i, question in enumerate(sample_df["question"].tolist()):
    print(f"[{i+1}/817] Generating...")
    answers.append(generate_answer(question))

sample_df["generated_answer"] = answers
sample_df[["question", "best_answer", "generated_answer"]]

In [ ]:
# CSV File Download Stage 2
sample_df.to_csv("stage2_output.csv", index=False)
print(" Saved! Download it from the files -> content panel on the left.")

In [ ]:
# Labeling
import pandas as pd
from sentence_transformers import SentenceTransformer, util

sample_df = pd.read_csv("stage2_output.csv")

sim_model = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_label(generated, truth):
    emb1 = sim_model.encode(generated, convert_to_tensor=True)
    emb2 = sim_model.encode(truth, convert_to_tensor=True)
    score = util.cos_sim(emb1, emb2).item()
    return 1 if score >= 0.5 else 0

sample_df["label"] = [
    semantic_label(row["generated_answer"], row["best_answer"])
    for _, row in sample_df.iterrows()
]

print(f"Labeled {len(sample_df)} samples. Hallucination rate: {1 - sample_df['label'].mean():.1%}")
sample_df[["question", "best_answer", "generated_answer", "label"]].head()

In [ ]:
# CSV File Download Stage 3
sample_df.to_csv("stage3_output.csv", index=False)
print(" Saved! Download it from the files -> content panel on the left.")

In [ ]:
# Halluciation Classifier
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import pandas as pd

stage3_df = pd.read_csv("stage3_output.csv")
stage3_df["text"] = stage3_df["question"] + " " + stage3_df["generated_answer"]

train_df, val_df = train_test_split(stage3_df, test_size=0.2, random_state=42)

clf_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
base_model = AutoModel.from_pretrained("distilbert-base-uncased")

class HallucinationClassifier(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.bert = base_model
        self.pre_classifier = nn.Linear(768, 768)
        self.classifier = nn.Linear(768, 2)
        self.dropout = nn.Dropout(0.3)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state[:, 0]
        hidden = self.pre_classifier(hidden)
        hidden = torch.nn.functional.relu(hidden)
        hidden = self.dropout(hidden)
        return self.classifier(hidden)

clf_model = HallucinationClassifier(base_model)
device = "cuda" if torch.cuda.is_available() else "cpu"
clf_model.to(device)

print(f" Classifier built on {device}. Train: {len(train_df)} | Val: {len(val_df)}")


In [ ]:
# Train/Val Spilt & Tokenisation
class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.encodings = tokenizer(
            df["text"].tolist(),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.tensor(df["label"].tolist())

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "label": self.labels[idx]
        }

train_dataset = QADataset(train_df, clf_tokenizer)
val_dataset   = QADataset(val_df, clf_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)

print(f" Datasets tokenized. Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:
#Training step
optimizer = torch.optim.AdamW(clf_model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):
    clf_model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        logits = clf_model(input_ids, attention_mask)
        loss   = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    clf_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            logits = clf_model(input_ids, attention_mask)
            preds  = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_acc = correct / total
    print(f"Epoch {epoch+1}/3 | Loss: {avg_train_loss:.4f} | Val Accuracy: {val_acc:.3f}")

print("✅ Classifier trained.")

Epoch 1/3 | Loss: 0.4809 | Val Accuracy: 0.835
Epoch 2/3 | Loss: 0.3332 | Val Accuracy: 0.866


In [ ]:
def predict_confidence(question, answer):
    inputs = clf_tokenizer(
        question + " " + answer,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256
    ).to(device)

    clf_model.eval()
    with torch.no_grad():
        logits = clf_model(inputs["input_ids"], inputs["attention_mask"])
    probs = torch.softmax(logits, dim=1)
    return probs[0][1].item()

sample_q = stage3_df["question"].iloc[0]
sample_a = stage3_df["generated_answer"].iloc[0]
print(f"Q: {sample_q}")
print(f"A: {sample_a}")
print(f"Confidence (correct): {predict_confidence(sample_q, sample_a):.3f}")

In [ ]:
# Experiment Log - For Classifer Training
import pandas as pd

experiment_log = pd.DataFrame([{
    "experiment_number": 1,
    "base_model": clf_model.bert.config.name_or_path,
    "classifier_architecture": (
        f"Linear({clf_model.pre_classifier.in_features}→{clf_model.pre_classifier.out_features})"
        f" → ReLU → Dropout({clf_model.dropout.p})"
        f" → Linear({clf_model.classifier.in_features}→{clf_model.classifier.out_features})"
    ),
    "optimizer": type(optimizer).__name__,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "loss_function": type(loss_fn).__name__,
    "epochs": epoch + 1,
    "batch_size": train_loader.batch_size,
    "max_token_length": train_dataset.encodings["input_ids"].shape[1],
    "train_test_split": f"{len(train_df)}/{len(val_df)}",
    "labeling_model": sim_model.get_sentence_embedding_dimension() and "all-MiniLM-L6-v2",
    "labeling_threshold": 0.5,
    "dataset_size": len(stage3_df),
    "train_size": len(train_df),
    "val_size": len(val_df),
    "hallucination_rate": f"{1 - stage3_df['label'].mean():.1%}",
    "final_val_accuracy": round(val_acc, 4),
    "sample_confidence_score": round(predict_confidence(sample_q, sample_a), 4),
}])

try:
    existing = pd.read_csv("experiment_log.csv")
    experiment_log["experiment_number"] = existing["experiment_number"].max() + 1
    updated = pd.concat([existing, experiment_log], ignore_index=True)
    updated.to_csv("experiment_log.csv", index=False)
    print(f"✅ Experiment {experiment_log['experiment_number'].values[0]} appended to log.")
    updated
except FileNotFoundError:
    experiment_log.to_csv("experiment_log.csv", index=False)
    print("✅ Experiment log created.")
    experiment_log

In [ ]:
# run each question through the generator then check if its a hallucination
THRESHOLD = 0.6

def run_inference(question):
    answer = generate_answer(question)
    score  = predict_confidence(question, answer)

    return {
        "question"                : question,
        "generated_answer"        : answer,
        "confidence"              : round(score, 3),
        "flagged_as_hallucination" : score < THRESHOLD
    }
results = []
for i, q in enumerate(stage3_df["question"].tolist()):
    results.append(run_inference(q))
    if (i+1) % 100 == 0:
        print(f"{i+1}/817 done")

inference_df = pd.DataFrame(results)
inference_df.head()

In [ ]:
flagged = inference_df["flagged_as_hallucination"].sum()
print(f"flagged{flagged} out of {len(inference_df)}")
print(f"hallucination rate:{flagged/len(inference_df):.1%}")


In [ ]:
inference_df.to_csv("stage5_output.csv", index=False)
print("saved!")

In [ ]:
# drop duplicates and keep original 817 + the 694 new ones only
stage3_df = stage3_df.drop_duplicates(subset=["question"]).reset_index(drop=True)
stage3_df.to_csv("stage3_output.csv", index=False)
print(f"cleaned! rows: {len(stage3_df)}")

In [ ]:
# Stage 6 - Feedback Loop
import pandas as pd

# grab only the flagged ones
flagged_df = inference_df[inference_df["flagged_as_hallucination"] == True].copy()
print(f"flagged rows: {len(flagged_df)}")

# relabel using semantic similarity to simulate human review
flagged_df["label"] = flagged_df.apply(
    lambda row: semantic_label(row["generated_answer"],
    stage3_df.loc[stage3_df["question"] == row["question"], "best_answer"].values[0]
    if len(stage3_df.loc[stage3_df["question"] == row["question"]]) > 0 else ""),
    axis=1
)
print(f"relabeled! correct: {flagged_df['label'].sum()} | hallucinations: {(flagged_df['label']==0).sum()}")

# merge flagged samples back into training data
updated_df = pd.concat([stage3_df, flagged_df[["question", "generated_answer", "label"]]], ignore_index=True)
updated_df.to_csv("stage3_output.csv", index=False)
print(f"training set grew from {len(stage3_df)} to {len(updated_df)} samples")

# reload and retrain on updated data
stage3_df = pd.read_csv("stage3_output.csv")
stage3_df["text"] = stage3_df["question"] + " " + stage3_df["generated_answer"]

train_df, val_df = train_test_split(stage3_df, test_size=0.2, random_state=42)
train_dataset = QADataset(train_df, clf_tokenizer)
val_dataset   = QADataset(val_df, clf_tokenizer)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=16)

base_model_v2 = AutoModel.from_pretrained("distilbert-base-uncased")
clf_model     = HallucinationClassifier(base_model_v2).to(device)
optimizer     = torch.optim.AdamW(clf_model.parameters(), lr=2e-5)
loss_fn       = nn.CrossEntropyLoss()

for epoch in range(3):
    clf_model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)
        optimizer.zero_grad()
        logits = clf_model(input_ids, attention_mask)
        loss   = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    clf_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)
            preds          = torch.argmax(clf_model(input_ids, attention_mask), dim=1)
            correct       += (preds == labels).sum().item()
            total         += labels.size(0)

    print(f"epoch {epoch+1}/3 | loss: {total_loss/len(train_loader):.4f} | val acc: {correct/total:.4f}")

print("retrain done!")

In [ ]:
# Stage 7 — Evaluation (evaluation/metrics.py)
# Measures classifier performance with accuracy, F1, precision, recall
# and compares BEFORE vs AFTER the feedback loop.

import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Helper: run the current clf_model over a DataLoader
def evaluate_model(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)
            logits         = model(input_ids, attention_mask)
            preds          = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    return all_labels, all_preds

# Helper: compute & print a metrics dict
def compute_metrics(labels, preds, tag=""):
    acc  = accuracy_score(labels, preds)
    f1   = f1_score(labels, preds, average="binary", zero_division=0)
    prec = precision_score(labels, preds, average="binary", zero_division=0)
    rec  = recall_score(labels, preds, average="binary", zero_division=0)
    print(f"\n{'='*50}")
    print(f"  {tag}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print("\nClassification Report:")
    print(classification_report(labels, preds,
                                target_names=["Hallucination","Correct"],
                                zero_division=0))
    return {"accuracy": acc, "f1": f1, "precision": prec, "recall": rec}



#  BEFORE feedback loop  →  use the val split from Stage 4
#  (val_df / val_loader were set in Cell 9 and are still in scope)
print("Evaluating BEFORE feedback loop (original val set, Stage 4 model)...")

# Rebuild val_loader from the pre-feedback val split
# (val_df is the 20 % split created in Cell 9 before Stage 6 ran)
stage3_original = pd.read_csv("stage5_output.csv")   # inference results on original 817

# Reconstruct ground-truth labels for those rows from stage3_output
# Use the label column that was assigned by the semantic similarity labeler
original_labeled = pd.read_csv("stage3_output.csv")
merged = stage3_original.merge(
    original_labeled[["question", "label"]].drop_duplicates("question"),
    on="question", how="left"
)
merged["label"] = merged["label"].fillna(0).astype(int)

# Predicted label: 1 = correct (not hallucination), 0 = hallucination
merged["pred_label"] = (merged["flagged_as_hallucination"] == False).astype(int)

before_metrics = compute_metrics(
    merged["label"].tolist(),
    merged["pred_label"].tolist(),
    tag="BEFORE Feedback Loop"
)


#  AFTER feedback loop  →  re-run clf_model (retrained in Stage 6)
#  over the same 817 questions
print("\nEvaluating AFTER feedback loop (retrained model)...")

after_results = []
for _, row in merged.iterrows():
    score = predict_confidence(row["question"], row["generated_answer"])
    after_results.append(1 if score >= 0.6 else 0)

after_metrics = compute_metrics(
    merged["label"].tolist(),
    after_results,
    tag="AFTER Feedback Loop"
)


#  Side-by-side comparison bar chart
metrics_names = ["Accuracy", "F1 Score", "Precision", "Recall"]
before_vals   = [before_metrics["accuracy"], before_metrics["f1"],
                 before_metrics["precision"], before_metrics["recall"]]
after_vals    = [after_metrics["accuracy"],  after_metrics["f1"],
                 after_metrics["precision"],  after_metrics["recall"]]

x     = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars_before = ax.bar(x - width/2, before_vals, width, label="Before Feedback Loop",
                     color="#4C72B0", alpha=0.85)
bars_after  = ax.bar(x + width/2, after_vals,  width, label="After Feedback Loop",
                     color="#55A868", alpha=0.85)

ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("Hallucination Detector — Before vs. After Feedback Loop")
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()

for bar in bars_before:
    ax.annotate(f"{bar.get_height():.3f}",
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)
for bar in bars_after:
    ax.annotate(f"{bar.get_height():.3f}",
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("stage7_before_vs_after.png", dpi=150)
plt.show()
print("\n📊 Chart saved to stage7_before_vs_after.png")


#  Summary comparison table
summary = pd.DataFrame({
    "Metric"  : metrics_names,
    "Before"  : [f"{v:.4f}" for v in before_vals],
    "After"   : [f"{v:.4f}" for v in after_vals],
    "Delta"   : [f"{a-b:+.4f}" for a, b in zip(after_vals, before_vals)]
})
print("\nSummary Table:")
print(summary.to_string(index=False))
summary.to_csv("stage7_metrics_summary.csv", index=False)
print("\nStage 7 complete — metrics saved to stage7_metrics_summary.csv")


In [ ]:
# Save the trained classifier model and tokenizer
torch.save(clf_model.state_dict(), "hallucination_classifier.pt")
clf_tokenizer.save_pretrained("hallucination_tokenizer")
print("✅ Model and tokenizer saved.")